# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tu-h-nguyn/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

Lane 2 — Refresh / Content Opportunity Scoring. Notebook này là bản dựng chi tiết của ma trận đặc trưng
cuối cùng dùng trong capstone, cùng ba bài test rò rỉ chạy trên chính nó.

## 1. Build the feature vector

Ý tưởng trung tâm: cửa sổ 90 ngày của snapshot **tách chính xác** thành ba đoạn 30 ngày

```text
impressions_90d = first30 (ngày 61-90) + prev30 (ngày 31-60) + last30 (ngày 1-30)
                  \______ biết được ____/  \_____ biết được __/  \___ CỬA SỔ NHÃN ___/
```

Nhờ đó một snapshot cắt ngang trở thành một bài toán dự báo thật: đứng ở ranh giới, chỉ dùng phần bên
trái, dự đoán phần bên phải. Toàn bộ việc dựng đặc trưng nằm trong `cp.build_frame()` —
`work/scripts/capstone_pipeline.py` — nên notebook, report và paper không thể lệch nhau.

In [1]:
# --- Bootstrap: chạy được cả ở local lẫn trên Colab ---
import os, sys, urllib.request

BRANCHES = [
    "https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-Machine-Learning-Internship/main",
    "https://raw.githubusercontent.com/tu-h-nguyn/FlyRank-Machine-Learning-Internship/claude/search-ranking-capstone-166z1j",
]

def _pipeline_dir() -> str:
    """Toàn bộ logic capstone nằm trong MỘT file: work/scripts/capstone_pipeline.py."""
    for p in ["../scripts", "work/scripts", "scripts", "../../work/scripts"]:
        if os.path.exists(os.path.join(p, "capstone_pipeline.py")):
            return p
    os.makedirs("work/scripts", exist_ok=True)
    for base in BRANCHES:
        try:
            urllib.request.urlretrieve(f"{base}/work/scripts/capstone_pipeline.py",
                                       "work/scripts/capstone_pipeline.py")
            return "work/scripts"
        except Exception:
            continue
    raise RuntimeError("Không tải được capstone_pipeline.py")

sys.path.insert(0, _pipeline_dir())
import numpy as np
import pandas as pd
import capstone_pipeline as cp

raw = cp.load_raw()
frame = cp.build_frame(raw)
d, population = cp.apply_population_filter(frame)
y = d["label_declined"].to_numpy()
groups = d["client_id"].to_numpy()
X = cp.design_matrix(d)

print(f"Population: {population['rows_modelled']:,} trang / {population['clients_modelled']} client "
      f"(lọc từ {population['rows_start']:,} dòng, ngưỡng impressions_prev_30d >= {cp.MIN_PREV_IMPRESSIONS})")
print(f"Base rate (tỷ lệ trang thực sự suy giảm > 20% trong 30 ngày kế tiếp): {y.mean():.4f}")

# Chứng minh phép tách là chính xác, không phải giả định
recon = (frame["impressions_first30"] + frame["impressions_prev_30d"]
         + frame["impressions_last_30d"] - frame["impressions_90d"]).abs()
print(f"Sai lệch lớn nhất của phép tách cửa sổ: {recon.max():.0f} (0 = chính xác tuyệt đối)")

print(f"\nMa trận đặc trưng: {X.shape[0]:,} dòng × {X.shape[1]} cột")
display(X.head())
print("\nThống kê nhanh 6 đặc trưng quan trọng nhất:")
display(X[["log_impr_prev30", "log_clicks_prev30", "prior_ctr", "prior_impr_trend_pct",
           "content_age_days_at_decision", "word_count"]].describe().round(2))

Population: 18,010 trang / 30 client (lọc từ 30,000 dòng, ngưỡng impressions_prev_30d >= 100)
Base rate (tỷ lệ trang thực sự suy giảm > 20% trong 30 ngày kế tiếp): 0.6155
Sai lệch lớn nhất của phép tách cửa sổ: 0 (0 = chính xác tuyệt đối)

Ma trận đặc trưng: 18,010 dòng × 24 cột


,log_impr_prev30,log_impr_first30,prior_impr_trend_pct,log_clicks_prev30,prior_ctr,prior_ctr_delta,prior_click_trend_pct,log_sessions_prev30,prior_session_trend_pct,sessions_per_1k_impr_prev30,...,search_volume,competition,cpc,has_keyword_data,content_type_feedly article,content_type_keyword article,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,6.895683,7.713785,-55.898123,2.639057,1.317123,0.691564,-7.142857,2.302585,50.000000,9.118541,...,10.0,0.67,2.05,1.0,0.0,1.0,0.0,0.0,1.0,0.0
1,8.685416,8.840001,-14.325029,0.693147,0.016906,-0.041031,-75.000000,1.098612,-50.000000,0.338123,...,90.0,0.01,0.05,1.0,0.0,1.0,1.0,0.0,0.0,0.0
2,8.714403,8.321422,48.150852,1.386294,0.049269,-0.121047,-57.142857,1.386294,-57.142857,0.492692,...,0.0,0.00,0.00,1.0,0.0,1.0,1.0,0.0,0.0,0.0
3,8.344505,8.273847,7.323297,2.890372,0.404184,-0.080633,-10.526316,3.295837,52.941176,6.181645,...,10.0,0.00,0.00,1.0,0.0,1.0,0.0,0.0,0.0,0.0
4,8.772300,9.045230,-23.888168,1.098612,0.030998,-0.110561,-83.333333,2.302585,-92.622951,1.394916,...,0.0,0.00,0.00,1.0,0.0,1.0,1.0,0.0,0.0,0.0



Thống kê nhanh 6 đặc trưng quan trọng nhất:


,log_impr_prev30,log_clicks_prev30,prior_ctr,prior_impr_trend_pct,content_age_days_at_decision,word_count
count,18010.00,18010.00,18010.00,18010.00,18010.00,18010.00
mean,6.84,1.13,0.25,-0.97,228.89,3283.01
std,1.42,1.26,0.38,75.27,135.92,1289.11
min,4.62,0.00,0.00,-98.83,60.00,692.00
25%,5.68,0.00,0.00,-45.68,101.00,2760.00
50%,6.68,0.69,0.11,-21.25,206.00,2877.00
75%,7.81,1.79,0.35,14.16,317.00,3321.00
max,12.30,7.40,6.78,300.00,534.00,9546.00


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Đặc trưng | Nghĩa | Thiếu dữ liệu xử lý thế nào | Có trước thời điểm dự đoán? |
|---|---|---|---|
| `log_impr_prev30`, `log_impr_first30` | Mức nhu cầu ở hai cửa sổ nhìn thấy được (log1p vì đuôi rất nặng) | Không thiếu | CÓ |
| `log_clicks_prev30`, `log_sessions_prev30` | Click và session ở cửa sổ trước | Không thiếu | CÓ |
| `prior_impr_trend_pct`, `prior_click_trend_pct`, `prior_session_trend_pct` | Động lượng **trước** thời điểm quyết định (prev30 so với first30), cắt biên ở [-100, +300] | Mẫu số 0 → 0.0 | CÓ |
| `prior_ctr`, `prior_ctr_delta` | CTR cửa sổ trước (×100) và mức dịch chuyển của nó | Mẫu số 0 → 0.0 | CÓ |
| `sessions_per_1k_impr_prev30` | Nhu cầu chuyển thành session ở mức nào | Mẫu số 0 → 0.0 | CÓ |
| `impr_share_prev30` | Tỷ trọng nhu cầu của nửa gần hơn trong 60 ngày nhìn thấy được | Mẫu số 0 → 0.0 | CÓ |
| `content_age_days_at_decision` | Tuổi nội dung, trừ đi 30 ngày để lùi về đúng thời điểm quyết định | Không thiếu | CÓ |
| `word_count` + `has_word_count` | Độ dài bài, kèm cờ "có đo hay không" | Điền trung vị **sau khi** gắn cờ | CÓ |
| `search_volume`, `competition`, `cpc` + `has_keyword_data` | Ngữ cảnh từ khóa | Điền 0 **sau khi** gắn cờ | CÓ |
| `content_type`, `main_intent` (one-hot) | Loại nội dung và ý định tìm kiếm | `main_intent` thiếu → hạng mục `unknown` | CÓ |

**Vì sao phải có cờ `has_-`:** missingness ở tập này **có hệ thống, không ngẫu nhiên** — nó đi theo
`content_type` (một loại nội dung không có dữ liệu từ khóa nào cả). `fillna(0)` mù sẽ lén mã hóa loại
nội dung vào feature và mô hình sẽ học "0 nghĩa là loại X" thay vì học tín hiệu thật.

In [2]:
miss = pd.DataFrame({
    "Số dòng thiếu (dữ liệu thô)": raw[["word_count", "search_volume", "competition", "cpc",
                                        "main_intent"]].isna().sum(),
})
miss["% thiếu"] = (miss["Số dòng thiếu (dữ liệu thô)"] / len(raw) * 100).round(1)
display(miss)

print("Missingness của word_count theo content_type (bằng chứng cho thấy nó CÓ hệ thống):")
display((raw.groupby("content_type")["word_count"]
            .apply(lambda s: s.isna().mean() * 100).round(1)
            .rename("% thiếu word_count").to_frame()))
print("Missingness của search_volume theo content_type:")
display((raw.groupby("content_type")["search_volume"]
            .apply(lambda s: s.isna().mean() * 100).round(1)
            .rename("% thiếu search_volume").to_frame()))
print("=> Đây chính là lý do dùng cờ has_word_count / has_keyword_data trước khi điền.")

,Số dòng thiếu (dữ liệu thô),% thiếu
word_count,7699,25.7
search_volume,2468,8.2
competition,2468,8.2
cpc,2468,8.2
main_intent,2374,7.9


Missingness của word_count theo content_type (bằng chứng cho thấy nó CÓ hệ thống):


,% thiếu word_count
content_type,
comparison article,0.0
feedly article,0.0
keyword article,28.3


Missingness của search_volume theo content_type:


,% thiếu search_volume
content_type,
comparison article,0.0
feedly article,100.0
keyword article,1.4


=> Đây chính là lý do dùng cờ has_word_count / has_keyword_data trước khi điền.


## 3. The leakage hunt

Ba loại rò rỉ, ba bài test. Bài test đầu tiên là bài test của chính **bộ đo**: nếu cố tình nhét cột
dẫn xuất từ nhãn vào mà điểm số *không* nhảy lên gần 1.0 thì bộ đo mới là thứ hỏng.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.25,
                                random_state=cp.RANDOM_SEED).split(X, y, groups))

def quick(Xm, tag):
    m = cp.make_model(cp.PRIMARY_MODEL); m.fit(Xm.iloc[tr], y[tr])
    s = m.predict_proba(Xm.iloc[te])[:, 1]
    print(f"{tag:<46} ROC-AUC {roc_auc_score(y[te], s):.4f} | "
          f"PR-AUC {average_precision_score(y[te], s):.4f} | "
          f"P@50 {cp.precision_at_k(s, y[te], 50):.3f}")
    return s

print("--- TEST 1: cột dẫn xuất từ nhãn ---")
quick(X, "Feature set hợp lệ (24 cột)")
X1 = X.copy(); X1["trend_pct_LEAKED"] = d["trend_pct"].values
quick(X1, "+ trend_pct (nhãn được tính từ cột này)")
print("   => Nhảy lên ~1.0 đúng như dự đoán: bộ đo hoạt động, và cột này vĩnh viễn bị cấm.\n")

print("--- TEST 2: cửa sổ tương lai / chồng lấn ---")
X2 = cp.design_matrix(d, extra_numeric=cp.LEAKY_EXTRA)
quick(X2, "+ các tổng và tỷ lệ 90 ngày")
print("   => impressions_90d - prev30 - first30 CHÍNH LÀ cửa sổ nhãn. Chồng lấn = rò rỉ.\n")

print("--- TEST 3: product flags (quyết định của hệ thống cũ) ---")
X3 = X.copy()
X3["impression_tier_ord"] = pd.Categorical(
    d["impression_tier"], categories=["no_data", "none", "low", "moderate", "good", "excellent"],
    ordered=True).codes
X3["freshness_tier_ord"] = pd.Categorical(
    d["freshness_tier"], categories=["never", "0-30", "31-90", "91-180", "181+"],
    ordered=True).codes
quick(X3, "+ impression_tier & freshness_tier (product buckets)")
print("   => Chúng nâng điểm vì được tính từ cột chồng cửa sổ. Chỉ dùng làm BASELINE, không làm input.")

--- TEST 1: cột dẫn xuất từ nhãn ---
Feature set hợp lệ (24 cột)                    ROC-AUC 0.6081 | PR-AUC 0.6652 | P@50 0.880


+ trend_pct (nhãn được tính từ cột này)        ROC-AUC 0.9997 | PR-AUC 0.9998 | P@50 1.000
   => Nhảy lên ~1.0 đúng như dự đoán: bộ đo hoạt động, và cột này vĩnh viễn bị cấm.

--- TEST 2: cửa sổ tương lai / chồng lấn ---


+ các tổng và tỷ lệ 90 ngày                    ROC-AUC 0.6710 | PR-AUC 0.7376 | P@50 0.980
   => impressions_90d - prev30 - first30 CHÍNH LÀ cửa sổ nhãn. Chồng lấn = rò rỉ.

--- TEST 3: product flags (quyết định của hệ thống cũ) ---
+ impression_tier & freshness_tier (product buckets) ROC-AUC 0.6430 | PR-AUC 0.7011 | P@50 0.920
   => Chúng nâng điểm vì được tính từ cột chồng cửa sổ. Chỉ dùng làm BASELINE, không làm input.


## 4. What I excluded and why

Danh sách đầy đủ (in ở cell dưới) được lưu ngay trong `work/scripts/capstone_pipeline.py` và được ghi
vào `work/outputs/capstone_metrics.json`, khóa `features_excluded` — nên người đọc paper kiểm tra được
mà không cần tin lời tôi.

Ba trường hợp đáng nói nhất:

- **`trend_pct` / `trend_direction`** — nhãn được tính từ chúng. Dùng chúng là kết quả vòng tròn.
- **`avg_position`** — đây là tín hiệu SEO trung tâm, và tôi vẫn phải bỏ: cột duy nhất có sẵn là
  **trung bình 90 ngày**, chồng lên cửa sổ nhãn. Không có phiên bản "vị trí ở cửa sổ trước" nào trong
  slice này. Với dữ liệu daily của warehouse thì lấy lại được.
- **`days_since_last_update`** — 68.3% số trang được cập nhật *bên trong* cửa sổ kết quả. Cột này biết
  chuyện tương lai.

**Kiểm tra riêng tư (privacy):** không có tên khách hàng, domain, URL, tiêu đề hay truy vấn nào trong
bất kỳ đầu ra nào. `content_id` / `client_id` là mã giả danh và chỉ dùng để nhóm / split.

In [4]:
print(f"{len(cp.EXCLUDED)} nhóm cột bị loại:\n")
for col, why in cp.EXCLUDED.items():
    print(f"  {col:<52} {why}")

print("\n--- KIỂM TRA RIÊNG TƯ ---")
risky = [c for c in raw.columns
         if any(t in c.lower() for t in ["url", "domain", "title", "query", "email", "name", "text"])]
print(f"  Cột có thể nhận dạng trong dữ liệu thô: {risky if risky else 'không có'}")
print(f"  ID trong feature set: {[c for c in X.columns if 'id' == c[-2:]] or 'không có'}")
print(f"  Ví dụ ID (giả danh): {raw['content_id'].iloc[0]} / {raw['client_id'].iloc[0]}")

27 nhóm cột bị loại:

  trend_pct                                            label source (the label is derived from it) — circular
  trend_direction                                      label source — circular
  impressions_last_30d                                 inside the outcome window
  clicks_last_30d                                      inside the outcome window
  sessions_last_30d                                    inside the outcome window
  impressions_90d                                      90-day total spans the outcome window
  clicks_90d                                           90-day total spans the outcome window
  pageviews_90d                                        90-day total spans the outcome window
  sessions_90d                                         90-day total spans the outcome window
  users_90d                                            90-day total spans the outcome window
  engaged_sessions_90d                                 90-day total spans the out

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.